# 02 - Training Awal EfficientNet-B0

Notebook ini menjalankan eksperimen training awal untuk demo progress. Target sprint adalah pipeline training hidup dan menghasilkan metrik awal, bukan model final.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from cabai.config import CLASS_NAMES, RAW_DATA_DIR, INTERIM_DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, REPORTS_DIR
from cabai.data import build_manifest, assign_missing_splits, make_dataloaders, write_manifest_csv
from cabai.model import create_model, get_device
from cabai.train import set_seed, fit_model
from cabai.evaluate import predict_loader, classification_metrics, plot_history, plot_confusion_matrix, save_metrics_json

set_seed(42)
device = get_device()
print('Device:', device)
print('Classes:', CLASS_NAMES)

## Pilih Dataset Training

Prioritas: Roboflow sebagai training utama. Jika Roboflow belum ada, notebook otomatis memakai Kaggle sebagai fallback agar demo progress tetap punya training awal.

In [ ]:
roboflow_rows = build_manifest(RAW_DATA_DIR / 'roboflow_chili', dataset_name='roboflow_chili')
kaggle_rows = build_manifest(RAW_DATA_DIR / 'kaggle_penyakit_cabai', dataset_name='kaggle_penyakit_cabai')

if roboflow_rows:
    train_rows = roboflow_rows
    dataset_choice = 'roboflow_chili'
elif kaggle_rows:
    train_rows = kaggle_rows
    dataset_choice = 'kaggle_penyakit_cabai (fallback demo progress)'
else:
    raise FileNotFoundError(
        'Belum ada dataset lokal. Letakkan Roboflow di data/raw/roboflow_chili atau Kaggle di data/raw/kaggle_penyakit_cabai.'
    )

train_rows = assign_missing_splits(train_rows)
write_manifest_csv(train_rows, INTERIM_DATA_DIR / 'manifest_training.csv')
print('Dataset choice:', dataset_choice)
print('Total rows:', len(train_rows))

In [ ]:
import pandas as pd

df = pd.DataFrame(train_rows)
display(pd.crosstab(df['split'], df['label']))

## DataLoader dan Model

Model utama sprint: EfficientNet-B0 transfer learning. Jika `timm` belum tersedia, kode mencoba fallback torchvision agar smoke run tetap bisa dilakukan.

In [ ]:
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-4

loaders = make_dataloaders(train_rows, batch_size=BATCH_SIZE, num_workers=2)
print('Available splits:', list(loaders.keys()))

model = create_model('efficientnet_b0', num_classes=len(CLASS_NAMES), pretrained=True)

# Freeze backbone untuk demo training awal; classifier/head tetap dilatih.
for name, param in model.named_parameters():
    param.requires_grad = any(key in name.lower() for key in ['classifier', 'fc', 'head'])

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable_params:,} / {total_params:,}')

In [ ]:
checkpoint_path = CHECKPOINTS_DIR / 'efficientnet_b0_demo.pt'
history = fit_model(
    model,
    loaders,
    device=device,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    checkpoint_path=checkpoint_path,
)
print('Checkpoint:', checkpoint_path)

In [ ]:
fig = plot_history(history, FIGURES_DIR / 'training_curves.png')
fig

## Evaluasi Awal

Gunakan split `test` jika tersedia; jika belum ada, gunakan `val`. Metrik ini adalah hasil awal untuk demo progress.

In [ ]:
eval_split = 'test' if 'test' in loaders else 'val'
y_true, y_pred, y_prob, paths = predict_loader(model, loaders[eval_split], device)
metrics = classification_metrics(y_true, y_pred, CLASS_NAMES)
save_metrics_json(metrics, REPORTS_DIR / 'metrics_internal_demo.json')
print('Eval split:', eval_split)
print('Accuracy:', metrics['accuracy'])
print('Macro F1:', metrics['macro_f1'])

In [ ]:
fig = plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, FIGURES_DIR / 'confusion_matrix_internal.png')
fig

In [ ]:
preview = pd.DataFrame({
    'path': paths,
    'actual': [CLASS_NAMES[i] for i in y_true],
    'predicted': [CLASS_NAMES[i] for i in y_pred],
    'confidence': y_prob.max(axis=1),
})
display(preview.head(10))
preview.to_csv(REPORTS_DIR / 'prediction_preview_internal.csv', index=False)

## Catatan untuk Demo

- Jika hasil metrik belum tinggi, jelaskan bahwa ini training awal 5 epoch untuk membuktikan pipeline.
- Tahap final akan menambahkan tuning, evaluasi eksternal Kaggle, Grad-CAM, dan deployment web.